# Sentiment Analysis
## Working on Internet Movie Database (IMBd), building a predictor that can distinguish between positive and negative reviews

Obtaining the dataset
Were not abel to accsess http://ai.stanford.edu/~amaas/data/sentiment/ for some reason, so obtained the dataset from: https://huggingface.co/datasets/stanfordnlp/imdb instead

In [1]:
from datasets import load_dataset
ds = load_dataset("stanfordnlp/imdb")

In [2]:
import numpy as np
import pandas as pd

### Decompressing alle reviews:
In to one single dataframe object with 50 000 reveiews, with label/class 0 for negative and 1 for positive

In [3]:
# compressing all
df = pd.concat([ds["train"].to_pandas(), ds["test"].to_pandas()], ignore_index=True)
df = df.rename(columns={"text": "review", "label": "sentiment"})

# randomly shuffle
np.random.seed(0)
df = df.reindex(np.random.permutation(df.index))

# to csv file
df.to_csv("data/movie_data.csv", index=False, encoding="utf-8")


In [4]:
# checking if everything is correct
df = pd.read_csv("data/movie_data.csv", encoding="utf-8")
print(df.shape) 
df.head(3)

(50000, 2)


,review,sentiment
0,If your idea of entertainment is watching grap...,0
1,An intriguingly bold film weaves the seemingly...,1
2,"With ""Batman Returns"", Tim Burton succumbed to...",1


### Bag of words model
Categorical data such as text needs to be transformed into numerical form.
Bag-of-words lets us represent text as numerical vectors.
Idea behind bag-of-words:
1. Create a vocubalary of unique tokens (for example words) from the entire set of documents
2. Construct a feature vector from each document that contains the counts of how often each word occurs in the particular document

In [5]:
# use CountVectorizer to consturct a bag-of-words (can be used on array of text, entire documents, sentnces)
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer()

docs = np.array(["The sun is shining",
                 "The weather is sweet",
                 "The sun is shining, the weather is sweet"
                 "and one and one is two"])

# caling fit_transform, created vocubalary of the bag-of-words and transromed into three sentences into sparse vectors
bag = count.fit_transform(docs)

# prints contents, of the vocubulary
# stored in a dict
print(count.vocabulary_)

{'the': 7, 'sun': 4, 'is': 1, 'shining': 3, 'weather': 9, 'sweet': 5, 'sweetand': 6, 'one': 2, 'and': 0, 'two': 8}


In [6]:
# print the feature vectors of the sentences
print(bag.toarray())

[[0 1 0 1 1 0 0 1 0 0]
 [0 1 0 0 0 1 0 1 0 1]
 [1 3 2 1 1 0 1 2 1 1]]


This feature vectors are called the raw term frequencies tf(t,d), the number of time a term t, occurs in a document, d.
The bag-of-word model here is called 1-gram, because each item or token in the vocubulary represents a single word. In generell it's called n-grams model, the choice of n in n-gram model depends on the application. \
**Example**
- 1-gram: "the", "sun", "is", "shinning"
- 2-gram: "the sun", "sun is", "is shinning"

### Term frequency-inverse document frequency (tf-idf)
Used to downweight the frequently occuring words in the feature vectors. 

In [7]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf = TfidfTransformer(use_idf=True,
                         norm='l2',
                         smooth_idf=True)

np.set_printoptions(precision=2)

print(tfidf.fit_transform(count.fit_transform(docs)).toarray())

[[0.   0.43 0.   0.56 0.56 0.   0.   0.43 0.   0.  ]
 [0.   0.39 0.   0.   0.   0.66 0.   0.39 0.   0.5 ]
 [0.27 0.49 0.55 0.21 0.21 0.   0.27 0.32 0.27 0.21]]


The word "is" does not have a lower tf-idf (0.49) in the third documents, whereas before 3. This is since it's present in the first two documents also and thus unlikely to contain any usefull discriminatory information

### Cleaning text data
Before building the bag-of-word model the first most important step is to clean the data, by stripping it of all unwanted characters

In [ ]:
# illustrate why cleaning is important
df.loc[3456, 'review'][-50:]

# contains html markup, punctation and other letters

'r />from Andy Frye at MySportsComplex.blogspot.com'

In [30]:
# remove all puncation, except (emojis, emociton caracters (i.e. :)) because they are usefull for sentiment analysis
# using regex
import regex as re

def preprocessor(text):
    # try to remove all hmtl markup
    text = re.sub('<[^>]*>', '', text)

    # find all emotiocons
    emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)

    # remove all non-word char from text via regex [\W] + convert text into lowercase
    # added temporary stored emoticons to the end of the setence
    text = (re.sub('[\W]+', ' ', text.lower()) + ' '.join(emoticons).replace('-', ''))

    return text

In [33]:
# testing the preprocessor
print(preprocessor(df.loc[3456, 'review'][-50:]))

print(preprocessor("</a>This :) is :( a test :-)!"))

r from andy frye at mysportscomplex blogspot com
this is a test :) :( :)


In [34]:
# apply preprocessor
df['review'] = df['review'].apply(preprocessor)

### Processing documents into tokens
To split the text into individuals tokens. One way to tokenize documents is to split them into individual words by spliting the cleaned documents at their whitespace characters

In [35]:
def tokenizer(text):
    return text.split()

tokenizer('runners like running and thus they run')

['runners', 'like', 'running', 'and', 'thus', 'they', 'run']

In [36]:
# using stemming, which is to transforming the word to its root form, alows us to map related word to the same stem
from nltk.stem.porter import PorterStemmer

porter = PorterStemmer()

def tokenizer_porter(text):
    return [porter.stem(word) for word in text.split()]

tokenizer_porter('runners like running and thus they run')

['runner', 'like', 'run', 'and', 'thu', 'they', 'run']

In [41]:
# stop words
# words that are very common in all sorts of texts, bear littel to no usefull information that can be used to distinguish between classes/documents
# ex: is, has, like.
# can be usefull if working with raw or normalized term frequensies, rather than tf-idfs, which are alreade downweighting frequenlty occuring words

# use the set of 127 english stop-words
import nltk

nltk.download('stopwords')

# load and apply the stop words
from nltk.corpus import stopwords

stop = stopwords.words('english')

[w for w in tokenizer_porter('a runner likes running and runs a lot')[-10:] if w not in stop]

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/skramstad/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['runner', 'like', 'run', 'run', 'lot']

### Training a logistic regression model for document classification
Classify movie review into positive og negative reviews based on the bag-of-words model
25K for training and testing

In [42]:
X_train = df.loc[:25000, 'review'].values
y_train = df.loc[:25000, 'sentiment'].values

X_test = df.loc[25000:, 'review'].values
y_test = df.loc[25000:, 'sentiment'].values

In [48]:
# grid search, to find optimal set of parameter, using 5-fold stratified cross-validation
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(strip_accents=None,
                        lowercase=False,
                        preprocessor=None)

param_grid = [{'vect__ngram_range': [(1,1)],
               'vect__stop_words': [stop, None],
               'vect__tokenizer': [tokenizer, tokenizer_porter],
               'clf__penalty': ['l1', 'l2'],
               'clf__C': [1.0, 10.0, 100.0]},
               
               {'vect__ngram_range': [(1,1)],
               'vect__stop_words': [stop, None],
               'vect__tokenizer': [tokenizer, tokenizer_porter],
               'vect__use_idf': [False],
               'vect__norm': [None],
               'clf__penalty': ['l1', 'l2'],
               'clf__C': [1.0, 10.0, 100.0]}
               ]


lr_tfidf = Pipeline([('vect', tfidf),
                     ('clf', LogisticRegression(random_state=0, solver='liblinear'))])


gs_lr_tfidf = GridSearchCV(lr_tfidf, param_grid, scoring='accuracy', cv=5, verbose=2, n_jobs=-1)

gs_lr_tfidf.fit(X_train, y_train)

Fitting 5 folds for each of 48 candidates, totalling 240 fits


/home/skramstad/Documents/DAT300/.venv/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/skramstad/Documents/DAT300/.venv/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/skramstad/Documents/DAT300/.venv/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/skramstad/Documents/DAT300/.venv/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/skramstad/Documents/DAT300/.venv/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter '

[CV] END clf__C=1.0, clf__penalty=l1, vect__ngram_range=(1, 1), vect__stop_words=['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ou

/home/skramstad/Documents/DAT300/.venv/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/skramstad/Documents/DAT300/.venv/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


[CV] END clf__C=1.0, clf__penalty=l1, vect__ngram_range=(1, 1), vect__stop_words=['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ou

/home/skramstad/Documents/DAT300/.venv/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


... (383 outputs left out)

/home/skramstad/Documents/DAT300/.venv/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


[CV] END clf__C=100.0, clf__penalty=l1, vect__ngram_range=(1, 1), vect__norm=None, vect__stop_words=['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 

/home/skramstad/Documents/DAT300/.venv/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,estimator,Pipeline(step...liblinear'))])
,param_grid,"[{'clf__C': [1.0, 10.0, ...], 'clf__penalty': ['l1', 'l2'], 'vect__ngram_range': [(1, ...)], 'vect__stop_words': [['a', 'about', ...], None], ...}, {'clf__C': [1.0, 10.0, ...], 'clf__penalty': ['l1', 'l2'], 'vect__ngram_range': [(1, ...)], 'vect__norm': [None], ...}]"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [49]:
print('Best parameter set: %s ' % gs_lr_tfidf.best_params_)

Best parameter set: {'clf__C': 10.0, 'clf__penalty': 'l2', 'vect__ngram_range': (1, 1), 'vect__stop_words': None, 'vect__tokenizer': <function tokenizer at 0x7253c436ff40>} 


In [50]:
# using best model from grid search, lets print average 5-fold cross-validation accuracy scores on the training dataset and the classification accuracy on the dataset
print('CV Accuracy: %.3f' % gs_lr_tfidf.best_score_)

clf = gs_lr_tfidf.best_estimator_

print('Test accuracy: %.3f' % clf.score(X_test, y_test))

CV Accuracy: 0.895
Test accuracy: 0.898
